# Lecture 4 — Agentic Resume Outreach

**Goal**: Build an agent that scores resumes and drafts personalized outreach emails.

Our agent will:
1. **Score** each candidate using the optimized prompt from Lecture 3
2. **Decide** an outcome (interview, reject, or flag for review) based on the score
3. **Draft** a personalized email appropriate to the outcome
4. **Submit** the result to the leaderboard

## The Agent Loop
```
while not done:
    observe(action_history)
    decide(which_tool_to_call)
    execute(tool)        # ← tools make REAL LLM calls
    log(result)
```

The agent loop machinery lives in `agent_utils.py` — your job is to **define the tools** that the agent calls.

In [ ]:
import json
import random
import pandas as pd
from IPython.display import display, HTML

from resume_utils import load_resumes, load_job_requirements
from agent_utils import (
    structured_llm_call,
    run_agent,
    submit_outreach,
    delete_outreach,
    SCORING_PROMPT_TEMPLATE,
    ScoreResult,
    EmailResult,
)

random.seed(42)

# Configuration
OPENROUTER_API_KEY = ""  # <-- Paste your OpenRouter API key here
TEAM_NAME = ""           # <-- Your team name here
MODEL = "anthropic/claude-sonnet-4-6"

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError("Please set OPENROUTER_API_KEY above")
if not TEAM_NAME.strip():
    raise RuntimeError("Please set TEAM_NAME above")

# Load data
resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

# Build the scoring prompt
scoring_prompt = SCORING_PROMPT_TEMPLATE.format(job_req=job_req)

# Select candidates: 2 gold + 2 silver + 4 wild
gold = {rid: r for rid, r in resumes.items() if rid.startswith('g')}
silver = {rid: r for rid, r in resumes.items() if rid.startswith('s')}
wild = {rid: r for rid, r in resumes.items() if rid[0].isdigit()}
wild_sample = dict(random.sample(list(wild.items()), 4))

candidates = {**gold, **silver, **wild_sample}

## Step 1: Define Our Tools

Our agent has three tools. These make **real LLM calls** — the agent genuinely needs the loop because it can't draft an email until it knows the score.

| Tool | What it does | Why the agent needs it |
|------|-------------|----------------------|
| `score_resume` | Runs the L3 optimized scorer | Agent doesn't know the score until it calls this |
| `draft_outreach_email` | Generates a personalized email | Content depends on the score and outcome |
| `done` | Signals completion | Tells the loop to stop |

In [ ]:
# Tool 1: Score a resume using the Lecture 3 optimized prompt
def score_resume(candidate_id):
    """Score a candidate's resume. Returns score (0-100) and reasoning."""
    resume = resumes[candidate_id]
    result = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=scoring_prompt,
        context_data={"resume": resume['Resume_str']},
        output_schema=ScoreResult,
        model=MODEL,
        temperature=0.2,
    )
    if result['error']:
        return {"status": "error", "message": result['error']}
    
    score = result['result']['score']
    reasoning = result['result']['reasoning']
    return {
        "status": "success",
        "score": score,
        "reasoning": reasoning,
        "message": f"Score: {score}/100 — {reasoning}",
        "usage": result['usage'],
    }


# Tool 2: Draft a personalized outreach email
def draft_outreach_email(candidate_id, outcome, key_points):
    """Draft a professional email based on the routing outcome."""
    
    email_prompt = f"""Write a professional email from a hiring manager to a job candidate.

OUTCOME: {outcome}
KEY POINTS ABOUT THIS CANDIDATE: {key_points}

GUIDELINES:
- If INTERVIEW: Express enthusiasm about their background. Mention specific strengths from the key points. 
Invite them to schedule a technical interview. Keep it warm and professional.

- If REJECT: Be respectful and encouraging. Thank them for applying. 
Briefly note why they weren't selected (without being harsh). Wish them well.  

- If REVIEW: Explain that their application is being reviewed by the hiring team. 
Mention what caught your attention. Set expectations for next steps.

Write ONLY the email body (no subject line, no "Dear" salutation — just the body text). Keep it to 3-5 sentences. 
Be specific to this candidate — do not write a generic template."""

    result = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=email_prompt,
        context_data={},
        output_schema=EmailResult,
        model=MODEL,
        temperature=0.7,
    )
    if result['error']:
        return {"status": "error", "message": result['error']}
    
    email_body = result['result']['email_body']
    return {
        "status": "success",
        "email_body": email_body,
        "message": f"Email drafted ({len(email_body)} chars)",
        "usage": result['usage'],
    }


# Tool 3: Signal completion
def done(candidate_id):
    """Signal that processing is complete for this candidate."""
    return {"status": "success", "message": "Processing complete", "final": True}


# Build the tool registry
TOOL_REGISTRY = {
    "score_resume": {
        "function": score_resume,
        "description": "Score the candidate's resume against the job requirements. Returns a score (0-100) and reasoning. Call this FIRST.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID"
        }
    },
    "draft_outreach_email": {
        "function": draft_outreach_email,
        "description": "Draft a personalized outreach email based on the scoring outcome. Call this AFTER scoring.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID",
            "outcome": "string — must be INTERVIEW, REJECT, or REVIEW",
            "key_points": "string — key points about this candidate to mention in the email (from the score reasoning)"
        }
    },
    "done": {
        "function": done,
        "description": "Signal that all processing is complete for this candidate. Call this LAST, after scoring and drafting the email.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID"
        }
    },
}

print("Tools defined:")
for name, info in TOOL_REGISTRY.items():
    print(f"  - {name}: {info['description'][:60]}...")

## Step 2: Run the Agent

`run_agent()` (from `agent_utils.py`) runs the observe → decide → execute loop for each candidate. Watch the step-by-step output — the agent genuinely needs the loop because it can't draft an email until it knows the score.

Each result is submitted to the leaderboard automatically. View all teams at: http://ai-leaderboard.site/lecture4

In [ ]:
results = []

for cid in sorted(candidates.keys()):
    result = run_agent(
        api_key=OPENROUTER_API_KEY,
        candidate_id=cid,
        tool_registry=TOOL_REGISTRY,
        model=MODEL,
        temperature=0.3,
        verbose=True,
    )
    results.append(result)

    # Display the email
    tier = 'GOLD' if cid.startswith('g') else ('SILVER' if cid.startswith('s') else 'WILD')
    color = {'GOLD': '#b8860b', 'SILVER': '#708090', 'WILD': '#333'}[tier]
    outcome_color = {'INTERVIEW': '#27ae60', 'REJECT': '#e74c3c', 'REVIEW': '#f39c12'}.get(result['outcome'], '#666')
    html = f"""
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0; max-width:600px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="font-weight:bold; color:{color};">[{tier}] {cid}</span>
            <span style="background:{outcome_color}; color:white; padding:2px 10px; border-radius:12px; font-size:0.85em;">{result['outcome']}</span>
        </div>
        <div style="font-size:0.85em; color:#666; margin-bottom:8px;">Score: {result['score']}/100</div>
        <div style="white-space:pre-wrap; line-height:1.5;">{result['email_body'] or 'No email generated'}</div>
    </div>
    """
    display(HTML(html))

    # Submit to leaderboard
    if result['outcome'] and result['email_body']:
        resp = submit_outreach(
            team_name=TEAM_NAME,
            resume_id=cid,
            outcome=result['outcome'],
            email_text=result['email_body'],
            score=result['score'],
            cost=result.get('cost'),
        )
        status = "OK" if 'error' not in resp else resp['error']
        print(f"  Submitted: {status}")

print(f"\n{'='*70}")
print(f"Processed and submitted {len(results)} candidates for team '{TEAM_NAME}'")
print(f"View results at: http://ai-leaderboard.site/lecture4")
print(f"{'='*70}")